# Code Review Eval Guide

## Introduction

Code Reviews are an essential part of ensuring production code is tested, secure, and defect-free. AI coding tools like Codex can generate high-quality code reviews, and are rapidly augmenting - and even replacing - human code reviews. How can we prove that our AI code reviews are performing well?

This guide demonstrates how to build Evals for AI code review systems, starting with a simple system and evolving to use more sophisticated techniques:
  - Level 1: Simple Benchmark Eval
  - Level 2: Pairwise Comparison Eval
  - Level 3: Multi-turn Optimization Loop

Moving from "vibe reviewing" to evals enables faster experimentation and adoption of new models and reasoning levels, token usage optimization, and a higher level of trust in AI systems throughout the SDLC. 

TODO: Add links here to the example repo/notebook once ready

## Foundations

In order to construct a basic eval for code reviews, we're going to need the following pieces:
- a "golden dataset"
- a code reviewer model, the "system under test"
- a grader model, the "LLM-as-a-judge"
- a score aggregator

### The Golden Dataset

In the context of evals, a "golden dataset" is a trusted set of examples with expected answers/judgements that are used to measure model behavior. At a minimum, it should have:
1. **Inputs**: prompts, tasks, test cases, etc. 
2. **Outputs**: the correct answer, ideal behavior, annotataion, etc.

For this code review eval, we will start by using existing code reviews in the [openai/codex](https://github.com/openai/codex), the open-source repository for [Codex](https://openai.com/codex/). We can use the [GitHub CLI](https://cli.github.com) to retrieve a set of closed PRs with review comments. 

Thus, our golden dataset has the following:
1. **Inputs**: diff, code files modified, review comments
2. **Outputs**: review success (`True` if the PR was merged, `False` if it was not merged)

This dataset has enough for us to work on our eval, but we'll see how it can be improved as we evolve the example. 

### Code Reviewer Model

This is the "system under test", and is the model that we will be measuring and (soon) optimizing. 

In this simple example, we'll use `gpt-5.4` with the following prompt:

```markdown
You are an LLM code reviewer evaluating a pull request. Your job is to identify the most important issues in the diff, following the repository-specific instructions in AGENTS.md.

Output requirements:
- Return valid JSON only.
- Do not wrap the JSON in markdown fences.
- Follow this exact schema:

{
  "summary": "string",
  "comments": [
    {
      "path": "string",
      "line": 0,
      "severity": "low | medium | high",
      "category": "correctness | bug_risk | security | performance | testing | maintainability",
      "claim": "string",
      "suggestion": "string",
      "confidence": 0.0
    }
  ]
}
```

For now, we'll use an empty `AGENTS.md` file to focus on establishing a benchmark.

### Grader Model

This is the LLM which will be used to evaluate the output of the Code Reviewer model. Selecting an optimal grading model is beyond the scope of this guide. We will use `gpt-5.4-nano`, a low-latency model that is performant for classification tasks with clear instructions. We'll use the following prompt for grading:

```markdown
You are grading one LLM code review for a pull request.

Your task is to judge whether the generated review is acceptable overall.

Evaluation criteria:

1. Correctness
- Comments must be grounded in the actual diff.
- Comments must not invent nonexistent behavior, files, or risks.

2. Usefulness
- Comments should identify meaningful issues or omissions.
- Comments should be specific enough to help an engineer act.

3. Noise
- Penalize trivial style comments unless AGENTS.md explicitly prioritizes style.
- Penalize vague comments, redundant comments, or excessive low-value comments.
- A smaller set of strong comments is better than many weak comments.

4. Instruction adherence
- Judge the review against the provided AGENTS.md guidance while keeping correctness first.

How to use the human comments:
- Treat historical human comments as weak reference evidence, not absolute ground truth.
- They are useful context for whether the review noticed important concerns, but they are not the only valid source of truth.

Pass rule:
- Set `overall_pass` to `1` only when the review is acceptable overall for this pull request.
- A passing review should be grounded, useful, and not meaningfully noisy.

Output requirements:
- Return valid JSON only.
- Do not wrap the JSON in markdown fences.
- `noise=1` means the review is low-noise / acceptable on noise.
- Follow this exact schema:

{
  "correctness": 0,
  "usefulness": 0,
  "noise": 0,
  "overall_pass": 0,
  "reason": "string"
}

```

### Score Aggregator

We will use a simple average to aggregate the scores produced by the grader model:
```
pass_rate = sum(overall_pass) / pr_count
correctness_rate = sum(correctness) / pr_count
usefulness_rate = sum(usefulness) / pr_count
low_noise_rate = sum(noise) / pr_count
```

### The Eval Engine Loop

With all of our ingredients in place, the eval loop looks like:

```mermaid
flowchart TD
    A[Golden Data] --> | PR + Result | B[Reviewer Model]
    B --> | Code Review | C[Grader Model]        
    C --> A
    C --> |Score| D[Score Aggregator]
```



## Level 1: Simple Benchmark Eval

Now that we have our foundations in place, let's build the simple benchmark. 

### 1. Golden Dataset

The first step is to fetch PRs from `openai/codex`:
```bash
cd examples/evals/codereview_evals
pip install -e .
evalcr fetch-prs --repo openai/codex --limit 50
```

A cached PR snapshot includes the PR metadata, merge state, changed files, unified diff, issue comments, reviews, and inline review comments. At Level 1, merge state remains useful cohort metadata in the final report, while the grader is the actual scoring source.

### 2. Code Reviewer Model

The Level 1 harness keeps its reviewer defaults local to the harness:
- `1_benchmark_harness/eval_config.json` stores the reviewer and grader model names
- `1_benchmark_harness/AGENTS.md` stores the harness-local review guidance
- `1_benchmark_harness/reviewer_system.txt` stores the reviewer prompt
- `1_benchmark_harness/review_output_schema.json` stores the structured reviewer output schema

Because those defaults live with the harness, the run command stays short:

```bash
evalcr benchmark run --cache-key openai_codex --max-prs 5
```

### 3. Grader Model

The same harness folder also owns `grader_system.txt` and `grader_output_schema.json`. The grader sees the diff, the generated review, the harness-local `AGENTS.md`, and historical human comments as weak reference evidence before producing the binary Level 1 labels: `correctness`, `usefulness`, `noise`, and `overall_pass`.

### 4. Score Aggregator And Report

Each Level 1 run writes four artifacts into `1_benchmark_harness/results/<run_id>/`:
- `results.json`
- `results.csv`
- `summary.json`
- `report.html`

The report is a self-contained HTML file, so we can render the most recent one directly in the notebook:


In [ ]:
from pathlib import Path
from IPython.display import HTML, display

report_root = Path("evals/codereview_evals/1_benchmark_harness/results")
reports = sorted(
    path / "report.html"
    for path in report_root.iterdir()
    if path.is_dir() and (path / "report.html").exists()
)

if not reports:
    raise FileNotFoundError(
        "No Level 1 report found. Run `evalcr benchmark run --cache-key openai_codex --max-prs 5` first."
    )

display(HTML(reports[-1].read_text(encoding="utf-8")))
